# 2. Analyse a backtest

Statistics, concentration, held-versus-target weights, attribution, and charts.

Assumes you have read **`01_index_and_backtest.ipynb`** — the setup here is the
same and moves quickly, so the notebook can spend its length on the analysis.

Needs the plotting extra for the final section:

```
pip install "py-beacon[plot]"
```

## The two things worth reading carefully

**Held weights are not target weights.** Between rebalances the index holds
fixed units, so weights drift with relative performance. Attributing with
*targets* would credit a return the index did not earn to a position it did not
hold. `drifted_weights()` reconstructs what was actually held.

**Contributions sum to the total exactly.** Returns compound while
contributions add, so a naive sum undershoots. Carino linking corrects it, and
the residual at the end should sit at machine epsilon — anything larger means
an assumption broke upstream.

## Setup

In [ ]:
import logging
from pathlib import Path

import pandas as pd

from beacon.analysis import attribute, concentration, drifted_weights
from beacon.backtest.engine import BacktestEngine
from beacon.index.calculation import IndexCalculator
from beacon.index.constructor import IndexDefinition
from beacon.index.methodology import MarketCapWeighted
from beacon.synthetic import SyntheticConfig, generate

# The engine warns once per rebalance when a buy cannot be fully funded, which
# is interesting but seventeen lines of it buries the output. Drop this to
# WARNING to watch the partial fills happen.
logging.basicConfig(level=logging.ERROR,
                    format="%(levelname)s %(name)s: %(message)s")

pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

OUTPUT = Path("output")

## Step 1 — build something to analyse

Index and backtest in one cell, since notebook 01 covered them step by step.

In [ ]:
# Same universe as notebook 01; see there for why this seed.
CONFIG = SyntheticConfig(assets=40,
                         start="2021-01-04",
                         end="2024-12-31",
                         seed=3)

dataset = generate(CONFIG)
fetcher = dataset.fetcher()

definition = IndexDefinition(
    index_id="ANALYSE",
    index_name="ANALYSE Index",
    base_date=CONFIG.start,
    base_value=1000.0,
    currency=CONFIG.currency,
    eligibility_rules=[],
    weighting_scheme=MarketCapWeighted(use_free_float=True),
    rebalancing_frequency="QUARTERLY",
    universe_identifiers=list(dataset.universe.index),
    max_constituent_weight=0.10)

index = IndexCalculator(definition, fetcher).run(start_date=CONFIG.start,
                                                 end_date=CONFIG.end)

backtest = BacktestEngine(start_date=CONFIG.start,
                          end_date=CONFIG.end,
                          initial_capital=10_000_000.0,
                          data_provider=fetcher,
                          index_result=index,
                          transaction_cost_bps=10.0).run()

print(f"{len(dataset.universe)} names, {len(index.index_levels):,} days, "
      f"{len(index.weight_snapshots)} rebalances, "
      f"{len(backtest.portfolio.transactions):,} trades")

## Step 2 — summary statistics

In [ ]:
AS_PERCENT = {"total_return", "annualised_return", "volatility",
              "max_drawdown", "tracking_error", "tracking_difference"}

pd.Series({name: ("n/a" if value is None
                  else f"{value:.2%}" if name in AS_PERCENT
                  else f"{value:.3f}")
           for name, value in backtest.summary().items()},
          name="summary")

## Step 3 — how concentrated is it?

A count of constituents says nothing about concentration: forty names with one
at 40% is not a forty-name portfolio in any sense that matters.

**Effective names** is the answer to "how many *equally weighted* names would
be as concentrated as this?" — the reciprocal of the Herfindahl index. It
being below the real count is the entire point of the measure.

In [ ]:
targets = index.weight_snapshots[max(index.weight_snapshots)]
measures = concentration(targets)

pd.Series({"constituents": f"{measures.assets}",
           "herfindahl": f"{measures.herfindahl_index:.4f}",
           "effective names": f"{measures.effective_assets:.1f} of {measures.assets}",
           "largest weight": f"{measures.largest_weight:.2%}"},
          name="concentration")

## Step 4 — target weights versus held weights

The index rebalances quarterly. In between it holds fixed units, so the weights
it *actually* has drift away from the ones it set.

In [ ]:
prices = dataset.market.data["CLOSE"].unstack("IDENTIFIER")
held = drifted_weights(index.weight_snapshots, prices)

final = held.iloc[-1]

comparison = pd.DataFrame({"target": pd.Series(targets),
                           "held": final}).dropna()
comparison["drift"] = comparison["held"] - comparison["target"]

comparison.sort_values("drift", key=abs, ascending=False).head(10).style.format("{:.2%}")

In [ ]:
print(f"the drifts sum to {comparison['drift'].sum():+.4%}")
print("held weights renormalise to one, so what one name gained another lost")

The drift as it accumulates, for the four names that moved most:

In [ ]:
largest = comparison["drift"].abs().sort_values(ascending=False).index[:4]
held[largest].iloc[::42].style.format("{:.2%}")

## Step 5 — attribution

Which names produced the return, and how much did the mechanics cost?

Note the inputs: `held`, not `targets`. Attribution against target weights
would assign returns to positions the portfolio did not have.

`cap_drag` and `cost_drag` are passed in here as illustrative figures. They are
the two structural leaks — what the 10% cap cost by forcing money out of names
that outperformed, and what trading cost — and they belong in the decomposition
rather than being left to disappear into a residual.

In [ ]:
asset_returns = prices.pct_change().reindex(held.index)
period_returns = (held.shift(1) * asset_returns).sum(axis=1)

attribution = attribute(period_returns, held, asset_returns,
                        cap_drag=-0.003, cost_drag=-0.004)

rows = pd.DataFrame([{"asset": row.asset_id,
                      "contribution": row.contribution,
                      "average weight": row.average_weight,
                      "total return": row.total_return}
                     for row in attribution.contributions]
                    ).set_index("asset").sort_values("contribution",
                                                     ascending=False)

pd.concat([rows.head(5), rows.tail(3)]).style.format("{:.2%}")

### The reconciliation

This is the part to check. Contributions are Carino-linked, so they sum to the
*compounded* total rather than approximately to it.

In [ ]:
print(f"total return   {attribution.total_return:>+9.2%}")
print(f"sum of parts   {sum(r.contribution for r in attribution.contributions):>+9.2%}")
print(f"cap drag       {attribution.cap_drag:>+9.2%}")
print(f"cost drag      {attribution.cost_drag:>+9.2%}")
print(f"residual       {attribution.residual:>+9.2e}   <-- the check")

The residual is the honesty check on everything above. Returns compound
multiplicatively and contributions add, so a naive decomposition leaves a gap
that grows with the length of the run and the size of the returns. Carino
linking distributes that gap across the periods that caused it.

A residual at `1e-16` means the decomposition is exact. A residual at `1e-3`
would mean the numbers above are approximately right and should not be
reported to two decimal places.

## Step 6 — charts

Two ways: written to files, or drawn inline. Both use the same plotting
accessors hanging off the result objects.

In [ ]:
import matplotlib

matplotlib.use("Agg")

import matplotlib.pyplot as plt

from beacon.plot import use

use("light")
OUTPUT.mkdir(parents=True, exist_ok=True)

for name, draw in (("level", index.plot.level),
                   ("weights", index.plot.weights),
                   ("performance", backtest.plot.performance),
                   ("annual_returns", backtest.plot.annual_returns),
                   ("contributions", attribution.plot.contributions)):
    draw()
    plt.savefig(OUTPUT / f"{name}.png", dpi=110)
    plt.close("all")

print(f"wrote 5 charts to {OUTPUT.resolve()}")

Or inline, which is the point of a notebook:

In [ ]:
%matplotlib inline

use("light")
index.plot.level()
plt.show()

backtest.plot.performance()
plt.show()

attribution.plot.contributions()
plt.show()

## Where to go next

- **`03_index_futures.ipynb`** — price a future on the index you just built
- **`04_optimised_index.ipynb`** — replace the cap with real constraints